<a href="https://colab.research.google.com/github/drmalalaphd/NLP-Lingala-Dataset-from-Pop-The-Balloon-Congo-/blob/main/ep19_first_test_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lingala test run: Pop the Balloon, Episode 19 (audio from Drive)
Run the cells top to bottom. Before cell 3: Runtime > Change runtime type > T4 GPU.

In [2]:
!pip install -q faster-whisper "av<18"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/Into Binary Research Institute/Projects/Dataset Projects'
!ls -la "{FOLDER}"

# Name of the audio file you uploaded. Change it if the listing above shows a different name.
SRC = FOLDER + '/ep19.mp4'

# Cut the first 19 minutes (jingle included) and convert to 16 kHz mono wav
!ffmpeg -y -loglevel error -i "{SRC}" -t 1140 -vn -ac 1 -ar 16000 ep19.wav
!ls -la ep19.wav

Mounted at /content/drive
total 160997
drwx------ 2 root root      4096 Nov  6 04:23  .
drwx------ 3 root root      4096 Nov  6 04:23  ..
-rw------- 1 root root      3517 Nov  6 06:24  ep19_first_test_drive.ipynb
-rw------- 1 root root 164834994 Nov  6 06:18  ep19.mp4
-rw------- 1 root root      6303 Nov  6 04:27 'Pop the Balloon Dataset Sample (corrected)'
-rw------- 1 root root      6299 Nov  6 04:27 'Pop the Balloon Dataset Sample (old version)'
-rw------- 1 root root       182 Nov  6 04:38 'Pop the Balloon Source List.gsheet'
-rw-r--r-- 1 root root 36479010 Nov  6 06:51 ep19.wav


In [4]:
 import csv, os, subprocess
from faster_whisper import WhisperModel

URL = 'https://youtube.com/watch?v=ARak_ENt6ao'
LANG = None   # None = Whisper guesses. Later try 'ln' (Lingala) or 'fr' to compare.

model = WhisperModel('large-v3', device='cuda', compute_type='float16')
segments, info = model.transcribe('ep19.wav', language=LANG, vad_filter=True, beam_size=5)
print('Whisper language guess:', info.language, round(info.language_probability, 2))

os.makedirs('clips', exist_ok=True)
fields = ['clip_id','audio_file','source','start','end','transcript','whisper_language_guess',
          'language','speaker_id','speaker_gender','laughter','background_noise',
          'overlapping_speech','usable','checked_by']
rows = []
for i, seg in enumerate(segments, 1):
    cid = f'popballoon_ep19_{i:04d}'
    path = f'clips/{cid}.wav'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i','ep19.wav',
                    '-ss',str(seg.start),'-to',str(seg.end),path], check=True)
    rows.append({'clip_id':cid, 'audio_file':path, 'source':URL,
                 'start':round(seg.start,1), 'end':round(seg.end,1),
                 'transcript':seg.text.strip(), 'whisper_language_guess':info.language,
                 'language':'', 'speaker_id':'', 'speaker_gender':'', 'laughter':'',
                 'background_noise':'', 'overlapping_speech':'', 'usable':'', 'checked_by':''})

with open('ep19_clips.csv','w',newline='',encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(len(rows), 'clips saved')

RuntimeError: Library libcublas.so.12 is not found or cannot be loaded

In [ ]:
dest = FOLDER + '/ep19_first_test'
!mkdir -p "{dest}"
!cp -r clips ep19_clips.csv "{dest}"
print('Saved to', dest)